# XGBoost

### Import needed libraries

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb


### Initialize dataframes with aggregate data

In [4]:
phy30sdf = pd.read_csv('../../exports/data/phys_agg_30s.csv', encoding="utf-8-sig")
phy16sdf = pd.read_csv('../../exports/data/phys_agg_16s.csv', encoding="utf-8-sig")
phy10sdf = pd.read_csv('../../exports/data/phys_agg_10s.csv', encoding="utf-8-sig")

scada30sdf = pd.read_csv('../../exports/data/scada_resolved_agg_30s.csv', encoding="utf-8-sig")
scada16sdf = pd.read_csv('../../exports/data/scada_resolved_agg_16s.csv', encoding="utf-8-sig")
scada10sdf = pd.read_csv('../../exports/data/scada_resolved_agg_10s.csv', encoding="utf-8-sig")

# drop min_value and max_value columns from all dataframes
for df in [phy30sdf, phy16sdf, phy10sdf, scada30sdf, scada16sdf, scada10sdf]:
    df = df.drop(['min_value', 'max_value'], axis=1, errors='ignore', inplace=True)


## Some quick data analysis

#### Physical data

In [3]:
# break up rows with multiple attacks
# For the 30 second bucket size (physical)
phy30sdf['attack_types'] = phy30sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
phy30sdf = phy30sdf.explode('attack_types').reset_index(drop=True)
phy30sdf['attack_types'] = phy30sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [ ]:
# look at feature info
phy30sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17480 entries, 0 to 17479
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bucket            17480 non-null  object 
 1   system_id         17480 non-null  object 
 2   prop_key          17480 non-null  object 
 3   asset_id          17480 non-null  object 
 4   avg_value         17480 non-null  float64
 5   num_measurements  17480 non-null  int64  
 6   num_attacks       17480 non-null  int64  
 7   attack_types      17480 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 1.1+ MB


In [5]:
# The percent of the data that is an attack
print(f"phy30sdf: {(phy30sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

phy30sdf: 40.05% rows are attacks


In [6]:
# break up rows with multiple attacks
# For the 30 second bucket size (physical)
phy16sdf['attack_types'] = phy16sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
phy16sdf = phy16sdf.explode('attack_types').reset_index(drop=True)
phy16sdf['attack_types'] = phy16sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [ ]:
# look at feature info
phy16sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30120 entries, 0 to 30119
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bucket            30120 non-null  object 
 1   system_id         30120 non-null  object 
 2   prop_key          30120 non-null  object 
 3   asset_id          30120 non-null  object 
 4   avg_value         30120 non-null  float64
 5   num_measurements  30120 non-null  int64  
 6   num_attacks       30120 non-null  int64  
 7   attack_types      30120 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 1.8+ MB


In [8]:
# The percentage of the data that is attack
print(f"phy16sdf: {(phy16sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

phy16sdf: 30.15% rows are attacks


In [9]:
# break up rows with multiple attacks
# For the 30 second bucket size (physical)
phy10sdf['attack_types'] = phy10sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
phy10sdf = phy10sdf.explode('attack_types').reset_index(drop=True)
phy10sdf['attack_types'] = phy10sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [ ]:
# look at feature info
phy10sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46480 entries, 0 to 46479
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bucket            46480 non-null  object 
 1   system_id         46480 non-null  object 
 2   prop_key          46480 non-null  object 
 3   asset_id          46480 non-null  object 
 4   avg_value         46480 non-null  float64
 5   num_measurements  46480 non-null  int64  
 6   num_attacks       46480 non-null  int64  
 7   attack_types      46480 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 2.8+ MB


In [11]:
# The percentage of the data that is attack
print(f"phy10sdf: {(phy10sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")


phy10sdf: 26.33% rows are attacks


#### Network data

In [12]:
# break up rows with multiple attacks
# For the 30 second bucket size (network)
scada30sdf['attack_types'] = scada30sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
scada30sdf = scada30sdf.explode('attack_types').reset_index(drop=True)
scada30sdf['attack_types'] = scada30sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [ ]:
# look at feature info
scada30sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352738 entries, 0 to 352737
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   bucket                     352738 non-null  object 
 1   system_id                  352738 non-null  object 
 2   protocol                   352738 non-null  object 
 3   avg_size                   352738 non-null  float64
 4   source_total_packets       352738 non-null  int64  
 5   destination_total_packets  352738 non-null  int64  
 6   min_size                   352738 non-null  int64  
 7   max_size                   352738 non-null  int64  
 8   num_connections            352738 non-null  int64  
 9   source_ip                  352439 non-null  object 
 10  source_port                352352 non-null  float64
 11  source_mac                 352738 non-null  object 
 12  destination_ip             352439 non-null  object 
 13  destination_port           35

In [14]:
# The percentage of the data that is attack
print(f"scada30sdf: {(scada30sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

scada30sdf: 54.23% rows are attacks


In [15]:
# break up rows with multiple attacks
# For the 16 second bucket size (network)
scada16sdf['attack_types'] = scada16sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
scada16sdf = scada30sdf.explode('attack_types').reset_index(drop=True)
scada16sdf['attack_types'] = scada16sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [ ]:
# look at feature info
scada16sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352738 entries, 0 to 352737
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   bucket                     352738 non-null  object 
 1   system_id                  352738 non-null  object 
 2   protocol                   352738 non-null  object 
 3   avg_size                   352738 non-null  float64
 4   source_total_packets       352738 non-null  int64  
 5   destination_total_packets  352738 non-null  int64  
 6   min_size                   352738 non-null  int64  
 7   max_size                   352738 non-null  int64  
 8   num_connections            352738 non-null  int64  
 9   source_ip                  352439 non-null  object 
 10  source_port                352352 non-null  float64
 11  source_mac                 352738 non-null  object 
 12  destination_ip             352439 non-null  object 
 13  destination_port           35

In [17]:
# The percentage of the data that is attack
print(f"scada16sdf: {(scada16sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

scada16sdf: 54.23% rows are attacks


In [18]:
# break up rows with multiple attacks
# For the 10 second bucket size (network)
scada10sdf['attack_types'] = scada10sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
scada10sdf = scada30sdf.explode('attack_types').reset_index(drop=True)
scada10sdf['attack_types'] = scada10sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [ ]:
# look at feature info
scada10sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352738 entries, 0 to 352737
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   bucket                     352738 non-null  object 
 1   system_id                  352738 non-null  object 
 2   protocol                   352738 non-null  object 
 3   avg_size                   352738 non-null  float64
 4   source_total_packets       352738 non-null  int64  
 5   destination_total_packets  352738 non-null  int64  
 6   min_size                   352738 non-null  int64  
 7   max_size                   352738 non-null  int64  
 8   num_connections            352738 non-null  int64  
 9   source_ip                  352439 non-null  object 
 10  source_port                352352 non-null  float64
 11  source_mac                 352738 non-null  object 
 12  destination_ip             352439 non-null  object 
 13  destination_port           35

In [20]:
# The percentage of the data that is attack
print(f"scada10sdf: {(scada10sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

scada10sdf: 54.23% rows are attacks


## Classification model for the 30 second time frame physical data

### Extract feature and target array

In [21]:
# Drop num_attacks and attack_types columns from features
X, y = phy30sdf.drop('num_attacks', axis=1), phy30sdf[['attack_types']]

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse_output=False)
X_encoded = onehot_encoder.fit_transform(X)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


### Split the data

In [22]:
# We split the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into DMatrix

In [23]:
# Create DMatrix for XGBoost
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)

### Define hyperparameters

In [24]:
params = {
    "objective": "multi:softprob", # multi-class classification
    "tree_method": "hist",         # use histogram-based algorithm
    "num_class": 5                 # number of classes is 5
}


### Train the model

In [ ]:
n = 1000

model = xgb.train(
   params=params,     # training parameters
   dtrain=dtrain_clf, # training data
   num_boost_round=n, # number of boosting rounds
)

preds = model.predict(dtest_clf)

### Cross validate

In [ ]:
results = xgb.cv(
   params, dtrain_clf,   # parameters and training data
   num_boost_round=n,   # number of boosting rounds
   nfold=5,   # number of folds for cross-validation
   metrics=["mlogloss", "auc", "merror"],  # evaluation metrics
   early_stopping_rounds=50,   # stop if no improvement after 50 rounds
   as_pandas=True,   # return results as pandas DataFrame
   verbose_eval=10.  # verbose evaluation every 10 rounds
)

[0]	train-mlogloss:0.94323+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.94324+0.00001	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000
[49]	train-mlogloss:0.00020+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.00020+0.00000	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000


### Calculate performace metrics 

In [ ]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

# Calculate precision, recall, f1-score, and confusion matrix
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

# Print the results
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)

Precision: 1.000
Recall: 1.000
F1-score: 1.000
Confusion Matrix:
 [[ 190    0    0    0    0]
 [   0  424    0    0    0]
 [   0    0 3321    0    0]
 [   0    0    0  320    0]
 [   0    0    0    0  115]]


## Classification model for 16 second time frame physical data

### Extract feature and target array

In [28]:
# Drop num_attacks and attack_types columns from features
X, y = phy30sdf.drop('num_attacks', axis=1), phy30sdf[['attack_types']]

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse_output=False)
X_encoded = onehot_encoder.fit_transform(X)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


### Split the data

In [ ]:
# We split the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into DMatrix

In [ ]:
# Create DMatrix for XGBoost
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)


### Define hyperparameters

In [ ]:
params = {
    "objective": "multi:softprob", # multi-class classification
    "tree_method": "hist",         # use histogram-based algorithm
    "num_class": 5                 # number of classes is 5
}


### Train the model

In [ ]:
n = 1000

model = xgb.train(
   params=params,     # training parameters
   dtrain=dtrain_clf, # training data
   num_boost_round=n, # number of boosting rounds
)

preds = model.predict(dtest_clf)

### Cross validate

In [ ]:
results = xgb.cv(
   params, dtrain_clf,   # parameters and training data
   num_boost_round=n,   # number of boosting rounds
   nfold=5,   # number of folds for cross-validation
   metrics=["mlogloss", "auc", "merror"],  # evaluation metrics
   early_stopping_rounds=50,   # stop if no improvement after 50 rounds
   as_pandas=True,   # return results as pandas DataFrame
   verbose_eval=10.  # verbose evaluation every 10 rounds
)

[0]	train-mlogloss:0.94323+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.94324+0.00001	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000
[50]	train-mlogloss:0.00020+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.00020+0.00000	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000


### Calculate performace metrics

In [ ]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

# Calculate precision, recall, f1-score, and confusion matrix
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

# Print the results
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)

Precision: 1.000
Recall: 1.000
F1-score: 1.000
Confusion Matrix:
 [[ 190    0    0    0    0]
 [   0  424    0    0    0]
 [   0    0 3321    0    0]
 [   0    0    0  320    0]
 [   0    0    0    0  115]]


## Classification model for the 10 second time frame physical data

### Extract feature and target array

In [35]:
# Drop num_attacks and attack_types columns from features
X, y = phy10sdf.drop('num_attacks', axis=1), phy10sdf[['attack_types']]

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse_output=False)
X_encoded = onehot_encoder.fit_transform(X)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


### Split the data

In [ ]:
# We split the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into DMatrix

In [ ]:
# Create DMatrix for XGBoost
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)


### Define hyperparameters

In [ ]:
params = {
    "objective": "multi:softprob", # multi-class classification
    "tree_method": "hist",         # use histogram-based algorithm
    "num_class": 5                 # number of classes is 5
}


### Train the model

In [ ]:
n = 1000

model = xgb.train(
   params=params,       # training parameters
   dtrain=dtrain_clf,   # training data
   num_boost_round=n,   # number of boosting rounds
)

preds = model.predict(dtest_clf)

### Cross validate

In [ ]:
results = xgb.cv(
   params, dtrain_clf,  # parameters and training data
   num_boost_round=n,   # number of boosting rounds
   nfold=5,  # number of folds for cross-validation
   metrics=["mlogloss", "auc", "merror"], # evaluation metrics
   early_stopping_rounds=50,  # stop if no improvement after 50 rounds
   as_pandas=True, # return results as pandas DataFrame
   verbose_eval=10 # verbose evaluation every 50 rounds
)

[0]	train-mlogloss:0.94278+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.94279+0.00001	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000
[50]	train-mlogloss:0.00007+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.00007+0.00000	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000


### precision, recall, F1-score, confusion matrix 

In [41]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)

Precision: 1.000
Recall: 1.000
F1-score: 1.000
Confusion Matrix:
 [[ 412    0    0    0    0]
 [   0 1125    0    0    0]
 [   0    0 9191    0    0]
 [   0    0    0  755    0]
 [   0    0    0    0  137]]


## Classification model for the 30 second time frame network data

### Extract feature and target array

In [ ]:
# Drop num_attacks and attack_types columns from features
X, y = scada30sdf.drop('num_attacks', axis=1), scada30sdf[['attack_types']]

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse_output=False)
X_encoded = onehot_encoder.fit_transform(X)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


: 

### Split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into DMatrix

In [ ]:
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)


### Define hyperparameters

In [ ]:
params = {
    "objective": "multi:softprob",
    "tree_method": "hist",
    "num_class": 5
}


### Train the model

In [ ]:
n = 1000

model = xgb.train(
   params=params,
   dtrain=dtrain_clf,
   num_boost_round=n,
)

preds = model.predict(dtest_clf)

### Cross validate

In [ ]:
results = xgb.cv(
   params, dtrain_clf,
   num_boost_round=n,
   nfold=5,
   metrics=["mlogloss", "auc", "merror"],
   early_stopping_rounds=50,
   as_pandas=True,
   verbose_eval=50
)

[0]	train-mlogloss:0.94323+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.94324+0.00001	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000
[49]	train-mlogloss:0.00020+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.00020+0.00000	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000


### precision, recall, F1-score, confusion matrix 

In [ ]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)

Precision: 1.000
Recall: 1.000
F1-score: 1.000
Confusion Matrix:
 [[ 190    0    0    0    0]
 [   0  424    0    0    0]
 [   0    0 3321    0    0]
 [   0    0    0  320    0]
 [   0    0    0    0  115]]


## Classification model for the 16 second time frame network data

### Extract feature and target array

In [ ]:
# Drop num_attacks and attack_types columns from features
X, y = scada16sdf.drop('num_attacks', axis=1), scada16sdf[['attack_types']]

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse_output=False)
X_encoded = onehot_encoder.fit_transform(X)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


### Split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into DMatrix

In [ ]:
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)


### Define hyperparameters

In [ ]:
params = {
    "objective": "multi:softprob",
    "tree_method": "hist",
    "num_class": 5
}


### Train the model

In [ ]:
n = 1000

model = xgb.train(
   params=params,
   dtrain=dtrain_clf,
   num_boost_round=n,
)

preds = model.predict(dtest_clf)

### Cross validate

In [ ]:
results = xgb.cv(
   params, dtrain_clf,
   num_boost_round=n,
   nfold=5,
   metrics=["mlogloss", "auc", "merror"],
   early_stopping_rounds=50,
   as_pandas=True,
   verbose_eval=50
)

[0]	train-mlogloss:0.94323+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.94324+0.00001	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000
[49]	train-mlogloss:0.00020+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.00020+0.00000	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000


### precision, recall, F1-score, confusion matrix 

In [ ]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)

Precision: 1.000
Recall: 1.000
F1-score: 1.000
Confusion Matrix:
 [[ 190    0    0    0    0]
 [   0  424    0    0    0]
 [   0    0 3321    0    0]
 [   0    0    0  320    0]
 [   0    0    0    0  115]]


## Classification model for the 10 second time frame network data

### Extract feature and target array

In [ ]:
# Drop num_attacks and attack_types columns from features
X, y = scada10sdf.drop('num_attacks', axis=1), scada10sdf[['attack_types']]

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse_output=False)
X_encoded = onehot_encoder.fit_transform(X)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


### Split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into DMatrix

In [ ]:
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)


### Define hyperparameters

In [ ]:
params = {
    "objective": "multi:softprob",
    "tree_method": "hist",
    "num_class": 5
}


### Train the model

In [ ]:
n = 1000

model = xgb.train(
   params=params,
   dtrain=dtrain_clf,
   num_boost_round=n,
)

preds = model.predict(dtest_clf)

### Cross validate

In [ ]:
results = xgb.cv(
   params, dtrain_clf,
   num_boost_round=n,
   nfold=5,
   metrics=["mlogloss", "auc", "merror"],
   early_stopping_rounds=50,
   as_pandas=True,
   verbose_eval=50
)

[0]	train-mlogloss:0.94323+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.94324+0.00001	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000
[49]	train-mlogloss:0.00020+0.00000	train-auc:1.00000+0.00000	train-merror:0.00000+0.00000	test-mlogloss:0.00020+0.00000	test-auc:1.00000+0.00000	test-merror:0.00000+0.00000


### precision, recall, F1-score, confusion matrix 

In [ ]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)

Precision: 1.000
Recall: 1.000
F1-score: 1.000
Confusion Matrix:
 [[ 190    0    0    0    0]
 [   0  424    0    0    0]
 [   0    0 3321    0    0]
 [   0    0    0  320    0]
 [   0    0    0    0  115]]
